# SQLite Inspection

Detailed exploration notebook for `data/pystocks.sqlite`.

This notebook focuses on:
- schema and object inventory
- endpoint and series coverage
- dedupe/integrity checks for latest tables
- payload blob storage efficiency
- ingest run telemetry
- targeted deep dives by `conid`


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)


def _resolve_repo_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for root in candidates:
        if (root / "data" / "pystocks.sqlite").exists():
            return root
    return cwd


ROOT = _resolve_repo_root()
DB_PATH = ROOT / "data" / "pystocks.sqlite"
if not DB_PATH.exists():
    raise FileNotFoundError(f"SQLite database not found at: {DB_PATH}")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

print(f"Repo root: {ROOT}")
print(f"SQLite path: {DB_PATH}")


In [ ]:
def q(sql, params=None):
    params = params or []
    return pd.read_sql_query(sql, conn, params=params)


def table_columns(table_name):
    return q(f"PRAGMA table_info({table_name})")


def table_count(table_name):
    return int(q(f"SELECT COUNT(*) AS n FROM {table_name}").iloc[0]["n"])

q("select * from sqlite_master where type='table' and name like 'ratio%' order by name")
q("select * from sqlite_master where type='table' order by name")

In [ ]:
df = q("select * from sentiment_series")
df#.type.unique()#[['metric_id', 'value_num', 'avg_num']].groupby('metric_id').mean()

In [ ]:
q("select * from profile_and_fees where conid=756733")

In [ ]:
df = q("select * from lipper_ratings_values")
print(len(df))
df

In [ ]:
df = q("select * from morningstar_snapshots")
print(len(df))
df

In [ ]:
df = q("select * from morningstar_summary")
print(len(df))
df

In [ ]:
test = [756733, 45540646, 637692818, 89008395, 70600420, 689039014, 689038580, 89384980, 565109827, 802787801, 566361324, 75961307, 703680889, 521962253, 51529211, 625493486, 574926153, 105236041, 303019536, 334117506, 96674409, 308521989, 570487056, 52197301, 95931656, 538801521, 92214874, 294708502, 263755979, 407935212, 264398602, 143041840, 429910845, 556067829, 705912146, 705912151, 740522749, 702874693, 707033409, 459329475, 562595521]

[num for num in test if str(num) not in conids]

## Object Inventory


In [ ]:
len(df[df['debtors'].notna()]['conId'].to_list())

In [ ]:
conids = df[df['debtors'].notna()]['conId'].to_list()
conids = [str(e) for e in conids]

with open('/home/alex/Documents/pystocks/docs/bond_conids.txt', 'w') as f:
    f.write('\n'.join(conids))

In [ ]:
q("select * from landing_snapshots limit 50")#.columns

In [ ]:
objects = q(
    """
    SELECT *
    FROM sqlite_master
    where type = 'table'
    ORDER BY type, name
    """
)

display(objects[["type", "name"]])

tables = objects.loc[objects["type"] == "table", "name"].tolist()
views = objects.loc[objects["type"] == "view", "name"].tolist()

print(f"tables: {len(tables)}")
print(f"views:  {len(views)}")


In [ ]:
pragma_names = [
    "journal_mode",
    "synchronous",
    "foreign_keys",
    "temp_store",
    "wal_autocheckpoint",
]

pragma_rows = []
for name in pragma_names:
    value = q(f"PRAGMA {name}").iloc[0, 0]
    pragma_rows.append({"pragma": name, "value": value})

display(pd.DataFrame(pragma_rows))

if "schema_meta" in tables:
    display(q("SELECT * FROM schema_meta ORDER BY applied_at DESC"))
else:
    print("schema_meta table not found")


In [ ]:
counts = []
for table in tables:
    counts.append({"table": table, "rows": table_count(table)})

counts_df = pd.DataFrame(counts).sort_values(["rows", "table"], ascending=[False, True]).reset_index(drop=True)
display(counts_df)


## Endpoint Coverage


In [ ]:
snapshot_tables = sorted([t for t in tables if t.endswith("_snapshots")])

snapshot_rows = []
for table in snapshot_tables:
    row = q(
        f"""
        SELECT
            COUNT(*) AS snapshot_rows,
            COUNT(DISTINCT conid) AS conids,
            MIN(effective_at) AS min_effective_at,
            MAX(effective_at) AS max_effective_at,
            MAX(observed_at) AS last_observed_at
        FROM {table}
        """
    ).iloc[0]
    snapshot_rows.append(
        {
            "endpoint": table.replace("_snapshots", ""),
            "table": table,
            "snapshot_rows": int(row["snapshot_rows"]),
            "conids": int(row["conids"]),
            "min_effective_at": row["min_effective_at"],
            "max_effective_at": row["max_effective_at"],
            "last_observed_at": row["last_observed_at"],
        }
    )

snapshot_df = pd.DataFrame(snapshot_rows).sort_values(["snapshot_rows", "endpoint"], ascending=[False, True]).reset_index(drop=True)
display(snapshot_df)


In [ ]:
series_raw_tables = sorted([t for t in tables if t.endswith("_series_raw")])
series_latest_tables = sorted([t for t in tables if t.endswith("_series_latest")])

series_rows = []
for raw_table in series_raw_tables:
    base = raw_table[: -len("_series_raw")]
    latest_table = f"{base}_series_latest"
    raw_n = table_count(raw_table)
    latest_n = table_count(latest_table) if latest_table in series_latest_tables else 0
    ratio = (raw_n / latest_n) if latest_n else None
    series_rows.append(
        {
            "series": base,
            "raw_table": raw_table,
            "latest_table": latest_table,
            "raw_rows": raw_n,
            "latest_rows": latest_n,
            "raw_to_latest_ratio": ratio,
        }
    )

series_df = pd.DataFrame(series_rows).sort_values("series").reset_index(drop=True)
display(series_df)


## Storage and Telemetry


In [ ]:
if "raw_payload_blobs" in tables:
    blob_stats = q(
        """
        SELECT
            COUNT(*) AS blob_rows,
            SUM(raw_size_bytes) AS raw_bytes,
            SUM(compressed_size_bytes) AS compressed_bytes,
            ROUND(1.0 * SUM(compressed_size_bytes) / NULLIF(SUM(raw_size_bytes), 0), 4) AS compressed_to_raw_ratio,
            MIN(created_at) AS first_created_at,
            MAX(created_at) AS last_created_at
        FROM raw_payload_blobs
        """
    )
    display(blob_stats)
else:
    print("raw_payload_blobs table not found")


In [ ]:
if "ingest_runs" in tables:
    display(
        q(
            """
            SELECT
                run_id,
                run_started_at,
                run_finished_at,
                total_targeted_conids,
                processed_conids,
                saved_snapshots,
                inserted_events,
                overwritten_events,
                unchanged_events,
                series_raw_rows_written,
                series_latest_rows_upserted,
                auth_retries,
                aborted
            FROM ingest_runs
            ORDER BY run_id DESC
            LIMIT 20
            """
        )
    )

    if "ingest_run_endpoint_rollups" in tables:
        latest_run = q("SELECT run_id FROM ingest_runs ORDER BY run_id DESC LIMIT 1")
        if not latest_run.empty:
            run_id = int(latest_run.iloc[0]["run_id"])
            print(f"Endpoint rollups for run_id={run_id}")
            display(
                q(
                    """
                    SELECT
                        endpoint,
                        call_count,
                        useful_payload_count,
                        useful_payload_rate,
                        status_2xx,
                        status_4xx,
                        status_5xx,
                        status_other
                    FROM ingest_run_endpoint_rollups
                    WHERE run_id = ?
                    ORDER BY call_count DESC, endpoint
                    """,
                    [run_id],
                )
            )
else:
    print("ingest_runs table not found")


## Integrity Checks


In [ ]:
duplicate_checks = [
    ("price_chart_series_latest", "conid, trade_date"),
    ("sentiment_search_series_latest", "conid, datetime_ms"),
    (
        "ownership_trade_log_series_latest",
        "conid, trade_date, action, party, source, insider, shares, value, holding",
    ),
    (
        "dividends_events_series_latest",
        "conid, event_date, amount, currency, event_type, declaration_date, record_date, payment_date, description",
    ),
]

dup_rows = []
for table, key_cols in duplicate_checks:
    if table not in tables:
        dup_rows.append({"table": table, "duplicate_key_groups": None})
        continue

    dup_n = int(
        q(
            f"""
            SELECT COUNT(*) AS n
            FROM (
                SELECT {key_cols}, COUNT(*) AS c
                FROM {table}
                GROUP BY {key_cols}
                HAVING c > 1
            )
            """
        ).iloc[0]["n"]
    )
    dup_rows.append({"table": table, "duplicate_key_groups": dup_n})

display(pd.DataFrame(dup_rows))


In [ ]:
if "products" in tables:
    display(
        q(
            """
            SELECT
                COALESCE(last_status_fundamentals, '<null>') AS fundamentals_status,
                COUNT(*) AS products
            FROM products
            GROUP BY fundamentals_status
            ORDER BY products DESC, fundamentals_status
            """
        )
    )

    display(
        q(
            """
            SELECT
                COUNT(*) AS total_products,
                SUM(CASE WHEN last_scraped_fundamentals IS NOT NULL THEN 1 ELSE 0 END) AS scraped_once,
                MAX(last_scraped_fundamentals) AS latest_scrape_day
            FROM products
            """
        )
    )


## Conid Deep Dive


In [ ]:
TARGET_CONID = None

if snapshot_tables:
    union_sql = " UNION ALL ".join(
        [f"SELECT conid, '{table}' AS endpoint_table FROM {table}" for table in snapshot_tables]
    )
    coverage = q(
        f"""
        WITH endpoint_events AS (
            {union_sql}
        )
        SELECT
            conid,
            COUNT(DISTINCT endpoint_table) AS endpoint_tables,
            COUNT(*) AS snapshot_rows
        FROM endpoint_events
        GROUP BY conid
        ORDER BY endpoint_tables DESC, snapshot_rows DESC, conid
        LIMIT 20
        """
    )
    display(coverage)

    if TARGET_CONID is None and not coverage.empty:
        TARGET_CONID = str(coverage.iloc[0]["conid"])

print(f"TARGET_CONID: {TARGET_CONID}")


In [ ]:
if TARGET_CONID:
    if "products" in tables:
        display(q("SELECT * FROM products WHERE conid = ?", [TARGET_CONID]))

    if snapshot_tables:
        timeline_union = " UNION ALL ".join(
            [
                f"SELECT '{table}' AS endpoint_table, effective_at, observed_at, payload_hash FROM {table} WHERE conid = ?"
                for table in snapshot_tables
            ]
        )
        params = [TARGET_CONID] * len(snapshot_tables)
        timeline = q(
            f"""
            {timeline_union}
            ORDER BY observed_at DESC, endpoint_table
            """,
            params,
        )
        display(timeline.head(300))

        hash_changes = q(
            f"""
            WITH target_snapshots AS (
                {timeline_union}
            )
            SELECT
                endpoint_table,
                COUNT(*) AS rows,
                COUNT(DISTINCT payload_hash) AS distinct_payload_hashes,
                MIN(effective_at) AS min_effective_at,
                MAX(effective_at) AS max_effective_at
            FROM target_snapshots
            GROUP BY endpoint_table
            ORDER BY rows DESC, endpoint_table
            """,
            params,
        )
        display(hash_changes)


In [ ]:
if TARGET_CONID:
    series_queries = [
        (
            "price_chart_series_latest",
            "SELECT conid, trade_date, close, price, effective_at, observed_at FROM price_chart_series_latest WHERE conid = ? ORDER BY trade_date DESC LIMIT 50",
        ),
        (
            "sentiment_search_series_latest",
            "SELECT conid, trade_date, datetime_ms, sscore, sdelta, sbuzz, effective_at, observed_at FROM sentiment_search_series_latest WHERE conid = ? ORDER BY datetime_ms DESC LIMIT 50",
        ),
        (
            "dividends_events_series_latest",
            "SELECT conid, event_date, event_type, amount, currency, declaration_date, payment_date, effective_at FROM dividends_events_series_latest WHERE conid = ? ORDER BY event_date DESC LIMIT 50",
        ),
        (
            "ownership_trade_log_series_latest",
            "SELECT conid, trade_date, action, party, shares, value, holding, effective_at FROM ownership_trade_log_series_latest WHERE conid = ? ORDER BY trade_date DESC LIMIT 50",
        ),
    ]

    for table, sql in series_queries:
        if table in tables:
            print(table)
            display(q(sql, [TARGET_CONID]))


## Ad Hoc SQL

Use `q("...")` for additional checks, for example:
- `q("SELECT * FROM ratios_metrics LIMIT 50")`
- `q("SELECT endpoint, SUM(call_count) FROM ingest_run_endpoint_rollups GROUP BY endpoint ORDER BY 2 DESC")`

When finished, close the connection in a cell with `conn.close()`.
